In [ ]:
# Run from the repository root so that relative result paths resolve
# identically regardless of where this notebook lives.
from oddball_eeg.paths import use_project_root

PROJECT_ROOT = use_project_root()

In [ ]:
from __future__ import annotations

from pathlib import Path
import gc
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from scipy.stats import (
    mannwhitneyu,
    spearmanr,
    wilcoxon,
)

from IPython.display import display


# ------------------------------------------------------------
# Project modules
# ------------------------------------------------------------

from oddball_eeg.bids import make_epoching_config_from_experiment
from oddball_eeg.config import get_experiment, get_experiment_family
from oddball_eeg.loader import load_one_subject_grouped, load_subjects_grouped
from oddball_eeg.models import get_model
from oddball_eeg.preprocessing import (
    apply_crop_to_loaded_dataset,
    build_dataloaders,
    build_datasets_from_split,
    crop_time_window,
    make_class_weight_tensor,
    SplitArrays,
)
from oddball_eeg.reproducibility import get_default_device, seed_everything
from oddball_eeg.splits import build_purged_chunk_splits, make_group_train_val_split
from oddball_eeg.training import evaluate_eegnet_loader
from oddball_eeg.visualization import (
    round_numeric_columns,
    save_figure,
    set_pandas_display_options,
)


warnings.filterwarnings("ignore")
set_pandas_display_options()

DEVICE = get_default_device()

print("=" * 90)
print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print(
        "CUDA memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

print("=" * 90)

In [ ]:
# ------------------------------------------------------------
# Final experiment families
# ------------------------------------------------------------

EXPERIMENT_NAMES = [
    "ds003061_within_subject_max30_wide_timebin",
    "ds005863_within_subject_max30_wide_timebin",
    "ds006466_within_subject_max30_wide_timebin",
    "ds006480_within_subject_max30_wide_timebin",
]


# ------------------------------------------------------------
# Run stage
# ------------------------------------------------------------

# First use "pilot".
# After Cell 14 freezes the augmentation scale, change this to "full".
RUN_STAGE = "full"  # "pilot" or "full"

RUN_WITHIN_SUBJECT = False
RUN_LOSO = True

# Use None for every dataset.
# To run one dataset at a time, use for example:
# DATASET_FILTER = {"ds003061"}
DATASET_FILTER = None


# ------------------------------------------------------------
# Noise-scale selection
# ------------------------------------------------------------

# Sigma is applied after train-only channel standardization.
SIGMA_CANDIDATES = (0.01, 0.025, 0.05)

# Baseline sigma.
BASELINE_SIGMA = 0.0

# The validation improvement required to update an EEGNet checkpoint.
MIN_VAL_IMPROVEMENT = 1e-4


# ------------------------------------------------------------
# Validation-only pilot
# ------------------------------------------------------------

# Subjects are selected deterministically and approximately evenly across
# the configured subject list rather than simply taking the first subjects.
PILOT_SUBJECTS_PER_DATASET = 3
PILOT_FOLDS_PER_SUBJECT = 2
PILOT_SEEDS = (42,)


# ------------------------------------------------------------
# Full confirmatory ablation
# ------------------------------------------------------------

FULL_SEEDS = (42, 43, 44)

# Resume existing completed runs unless explicitly overwritten.
OVERWRITE_PILOT = False
OVERWRITE_FULL = False

# Continue after a failed run while recording the error.
# Set to False when debugging.
CONTINUE_ON_ERROR = True


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUT_ROOT = (
        PROJECT_ROOT
        / "results"
        / "thesis_artifacts"
        / "eegnet_gaussian_augmentation_ablation"
)

PILOT_ROOT = OUT_ROOT / "pilot"
PILOT_RUN_DIR = PILOT_ROOT / "run_records"
PILOT_HISTORY_DIR = PILOT_ROOT / "histories"
PILOT_TABLE_DIR = PILOT_ROOT / "tables"

FULL_ROOT = OUT_ROOT / "full"
FULL_RUN_DIR = FULL_ROOT / "run_records"
FULL_HISTORY_DIR = FULL_ROOT / "histories"
FULL_TABLE_DIR = FULL_ROOT / "tables"

FIG_DIR = OUT_ROOT / "figures"
ERROR_DIR = OUT_ROOT / "errors"

FROZEN_CHOICE_PATH = OUT_ROOT / "frozen_augmentation_choice.json"

QUALITY_INDEX_PATH = (
        PROJECT_ROOT
        / "results"
        / "thesis_artifacts"
        / "relative_importance_subject_quality_vs_classifier"
        / "tables"
        / "subject_quality_index.csv"
)

for directory in [
    PILOT_RUN_DIR,
    PILOT_HISTORY_DIR,
    PILOT_TABLE_DIR,
    FULL_RUN_DIR,
    FULL_HISTORY_DIR,
    FULL_TABLE_DIR,
    FIG_DIR,
    ERROR_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Plot defaults
# ------------------------------------------------------------

plt.style.use("default")
plt.rcParams.update({
    "figure.figsize": (9, 5.5),
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})


print("Run stage:", RUN_STAGE)
print("Within-subject:", RUN_WITHIN_SUBJECT)
print("LOSO:", RUN_LOSO)
print("Dataset filter:", DATASET_FILTER)
print("Candidate sigmas:", SIGMA_CANDIDATES)
print("Output root:", OUT_ROOT)

## Two-stage augmentation protocol

### Stage A: validation-only pilot

The pilot compares the baseline and three nonzero Gaussian-noise candidates using
only inner-validation ROC-AUC. Outer-test AUC is not calculated during this stage.

The pilot is used to freeze one nonzero value of \(\sigma\). The candidate with the
largest mean validation-AUC change relative to the no-augmentation baseline is
selected. Ties are resolved in favour of the smaller perturbation.

### Stage B: full confirmatory experiment

After the choice is frozen, the full analysis compares:

1. EEGNet without augmentation, \(\sigma=0\);
2. EEGNet with the frozen nonzero value of \(\sigma\).

Both conditions use identical:

- outer folds or held-out subjects;
- inner training and validation indices;
- model initialization seed;
- batch-order seed;
- optimizer and regularization settings;
- early-stopping rule;
- outer-test observations.

No full outer-test result should be used to change the frozen value of \(\sigma\).

In [ ]:
# ------------------------------------------------------------
# Label and path helpers
# ------------------------------------------------------------

def normalize_subject_id(value, dataset_label: str | None = None) -> str:
    if pd.isna(value):
        return np.nan

    subject = str(value).strip()

    if subject.startswith("sub-"):
        subject = subject[4:]

    if subject.endswith(".0"):
        subject = subject[:-2]

    label = str(dataset_label or "")

    if subject.isdigit():
        if label.startswith(("ds003061", "ds005863")):
            return subject.zfill(3)

        if label.startswith("ds006480"):
            return subject.zfill(2)

    return subject


def dataset_label_from_cfg(cfg) -> str:
    if cfg.dataset.dataset == "ds006480":
        return f"{cfg.dataset.dataset}_{cfg.dataset.event_preset}"

    return cfg.dataset.dataset


def display_dataset_label(value: str) -> str:
    replacements = {
        "ds006480_pooled": "ds006480 pooled",
    }
    return replacements.get(str(value), str(value))


def safe_file_token(value) -> str:
    token = str(value)
    for old, new in [
        (" ", "_"),
        ("/", "-"),
        ("\\", "-"),
        (":", "-"),
        (".", "p"),
    ]:
        token = token.replace(old, new)

    return token


def select_evenly_spaced(items, n_items: int):
    items = list(items)

    if n_items is None or n_items >= len(items):
        return items

    if n_items <= 0:
        return []

    indices = np.linspace(
        0,
        len(items) - 1,
        num=n_items,
    )
    indices = np.unique(np.round(indices).astype(int))

    return [items[i] for i in indices]


def make_run_id(metadata: dict, sigma: float) -> str:
    setting = (
        "within"
        if metadata["evaluation_setting"] == "Within-subject"
        else "loso"
    )

    return "__".join([
        setting,
        safe_file_token(metadata["dataset_label"]),
        f"sub-{safe_file_token(metadata['subject'])}",
        f"fold-{int(metadata['outer_fold']):02d}",
        f"seed-{int(metadata['base_seed']):03d}",
        f"sigma-{float(sigma):.3f}".replace(".", "p"),
    ])


# ------------------------------------------------------------
# JSON helpers
# ------------------------------------------------------------

def json_default(value):
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, torch.device):
        return str(value)

    raise TypeError(f"Cannot JSON-serialize object of type {type(value)}")


def save_json(payload: dict, path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_suffix(path.suffix + ".tmp")

    with temp_path.open("w", encoding="utf-8") as file:
        json.dump(
            payload,
            file,
            indent=2,
            sort_keys=True,
            default=json_default,
        )

    temp_path.replace(path)


def load_json_records(directory: str | Path) -> pd.DataFrame:
    directory = Path(directory)

    records = []
    for path in sorted(directory.glob("*.json")):
        try:
            with path.open("r", encoding="utf-8") as file:
                record = json.load(file)

            record["_record_path"] = str(path)
            records.append(record)

        except Exception as exc:
            print("Could not read:", path)
            print("Reason:", exc)

    return pd.DataFrame(records)


def write_error_record(
        *,
        stage: str,
        metadata: dict,
        sigma: float | None,
        error: Exception,
) -> None:
    timestamp = time.strftime("%Y%m%d-%H%M%S")

    path = (
            ERROR_DIR
            / f"{stage}_{timestamp}_{safe_file_token(metadata.get('dataset_label'))}.json"
    )

    payload = {
        "stage": stage,
        "sigma": sigma,
        "error_type": type(error).__name__,
        "error_message": str(error),
        **metadata,
    }

    save_json(payload, path)
    print("Error record saved:", path)

In [ ]:
# ------------------------------------------------------------
# Statistical helpers
# ------------------------------------------------------------

def bootstrap_mean_ci(
        values,
        *,
        confidence: float = 0.95,
        n_boot: int = 10_000,
        seed: int = 42,
) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)

    sampled_means = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )
        sampled_means[i] = sample.mean()

    alpha = 1.0 - confidence

    return (
        float(np.quantile(sampled_means, alpha / 2)),
        float(np.quantile(sampled_means, 1 - alpha / 2)),
    )


def matched_rank_biserial(delta) -> float:
    delta = np.asarray(delta, dtype=float)
    delta = delta[np.isfinite(delta)]
    delta = delta[delta != 0]

    if len(delta) == 0:
        return 0.0

    ranks = pd.Series(np.abs(delta)).rank(
        method="average"
    ).to_numpy()

    w_positive = ranks[delta > 0].sum()
    w_negative = ranks[delta < 0].sum()

    denominator = w_positive + w_negative

    if denominator == 0:
        return 0.0

    return float(
        (w_positive - w_negative) / denominator
    )


def safe_wilcoxon(delta) -> tuple[float, float, int]:
    delta = np.asarray(delta, dtype=float)
    delta = delta[np.isfinite(delta)]

    nonzero = delta[delta != 0]

    if len(nonzero) == 0:
        return 0.0, 1.0, int(len(delta))

    statistic, p_value = wilcoxon(
        nonzero,
        zero_method="wilcox",
        alternative="two-sided",
    )

    return (
        float(statistic),
        float(p_value),
        int(len(delta)),
    )


def safe_spearman(x, y) -> tuple[float, float, int]:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)

    x = x[mask]
    y = y[mask]

    if len(x) < 5:
        return np.nan, np.nan, int(len(x))

    if len(np.unique(x)) < 3 or len(np.unique(y)) < 3:
        return np.nan, np.nan, int(len(x))

    rho, p_value = spearmanr(x, y)

    return float(rho), float(p_value), int(len(x))


def holm_adjust(p_values) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)

    adjusted = np.full(
        shape=len(p_values),
        fill_value=np.nan,
        dtype=float,
    )

    finite_indices = np.where(np.isfinite(p_values))[0]

    if len(finite_indices) == 0:
        return adjusted

    finite_p = p_values[finite_indices]
    order = np.argsort(finite_p)

    running_max = 0.0
    m = len(finite_p)

    for rank, ordered_position in enumerate(order):
        raw_adjusted = (
                (m - rank)
                * finite_p[ordered_position]
        )

        running_max = max(running_max, raw_adjusted)

        adjusted[
            finite_indices[ordered_position]
        ] = min(running_max, 1.0)

    return adjusted

## Augmentation implementation

The existing preprocessing utilities calculate per-channel normalization statistics
from the inner training data and apply those statistics to the training, validation,
and test partitions.

Gaussian noise is added after this normalization step and only inside the training
loop:

```python
x_augmented = x_normalized + sigma * gaussian_noise

In [ ]:
# Cell 7 — Code: create one fixed inner split

# ------------------------------------------------------------
# Fixed inner train/validation split
# ------------------------------------------------------------

def make_inner_split_from_outer_partition(
        *,
        X_train_outer: np.ndarray,
        y_train_outer: np.ndarray,
        groups_train_outer: np.ndarray,
        X_test_outer: np.ndarray,
        y_test_outer: np.ndarray,
        cfg,
        training_seed: int,
) -> SplitArrays:
    train_inner_idx, val_inner_idx = (
        make_group_train_val_split(
            y_train=y_train_outer,
            groups_train=groups_train_outer,
            val_size=cfg.splits.val_size,
            seed=training_seed,
            n_tries=cfg.splits.val_n_tries,
        )
    )

    split = SplitArrays(
        X_train=np.asarray(
            X_train_outer[train_inner_idx],
            dtype=np.float32,
        ),
        y_train=np.asarray(
            y_train_outer[train_inner_idx],
            dtype=np.int64,
        ),
        X_val=np.asarray(
            X_train_outer[val_inner_idx],
            dtype=np.float32,
        ),
        y_val=np.asarray(
            y_train_outer[val_inner_idx],
            dtype=np.int64,
        ),
        X_test=np.asarray(
            X_test_outer,
            dtype=np.float32,
        ),
        y_test=np.asarray(
            y_test_outer,
            dtype=np.int64,
        ),
    )

    return split

In [ ]:
# ------------------------------------------------------------
# EEGNet training under one augmentation condition
# ------------------------------------------------------------

def scalar_metrics_only(metrics: dict | None) -> dict:
    if metrics is None:
        return {}

    keep = {}

    for key, value in metrics.items():
        if np.isscalar(value):
            keep[key] = float(value)

    return keep


def train_eegnet_condition(
        *,
        split: SplitArrays,
        cfg,
        sigma: float,
        training_seed: int,
        device,
        evaluate_test: bool,
        verbose: bool = False,
) -> dict:
    """
    Train EEGNet using one fixed augmentation condition.

    Parameters
    ----------
    split
        Fixed inner train/validation and outer test arrays.

    sigma
        Gaussian-noise SD after train-only channel normalization.
        sigma=0.0 is the unchanged baseline.

    evaluate_test
        False during pilot selection.
        True only after sigma has been frozen.
    """
    seed_everything(training_seed)

    ds_train, ds_val, ds_test, norm_stats = (
        build_datasets_from_split(
            split,
            normalize=cfg.eegnet.normalize,
        )
    )

    train_loader, val_loader, test_loader = (
        build_dataloaders(
            ds_train=ds_train,
            ds_val=ds_val,
            ds_test=ds_test,
            batch_size=cfg.eegnet.batch_size,
            device=device,
        )
    )

    eegnet_kwargs = {
        "n_classes": cfg.eegnet.n_classes,
        "f1": cfg.eegnet.f1,
        "d": cfg.eegnet.d,
        "f2": cfg.eegnet.f2,
        "k1": cfg.eegnet.k1,
        "k2": cfg.eegnet.k2,
        "dropout": cfg.eegnet.dropout,
    }

    model = get_model(
        "eegnet",
        n_ch=split.X_train.shape[1],
        n_times=split.X_train.shape[2],
        eegnet_kwargs=eegnet_kwargs,
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.eegnet.lr,
        weight_decay=cfg.eegnet.weight_decay,
    )

    if cfg.eegnet.use_class_weight:
        class_weight = make_class_weight_tensor(
            split.y_train,
            device=device,
        )
        criterion = nn.CrossEntropyLoss(
            weight=class_weight
        )
    else:
        criterion = nn.CrossEntropyLoss()

    # Separate random stream for augmentation noise.
    # Noise sampling therefore does not alter the batch-order RNG.
    noise_generator = torch.Generator(
        device=device
    )
    noise_generator.manual_seed(
        int(training_seed) + 1_000_000
    )

    history_rows = []

    best_val_auc = -np.inf
    best_epoch = None
    best_state = None

    patience_left = int(
        cfg.eegnet.early_stopping_patience
    )

    start_time = time.time()

    for training_epoch in range(
            1,
            int(cfg.eegnet.epochs) + 1,
    ):
        model.train()

        batch_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            if float(sigma) > 0:
                noise = torch.randn(
                    xb.shape,
                    generator=noise_generator,
                    device=xb.device,
                    dtype=xb.dtype,
                )

                xb_model = (
                        xb
                        + float(sigma) * noise
                )
            else:
                xb_model = xb

            logits = model(
                xb_model.unsqueeze(1)
            )

            loss = criterion(
                logits,
                yb,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            loss.backward()

            if (
                    cfg.eegnet.grad_clip_norm is not None
                    and cfg.eegnet.grad_clip_norm > 0
            ):
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    cfg.eegnet.grad_clip_norm,
                )

            optimizer.step()

            batch_losses.append(
                float(loss.item())
            )

        train_loss = (
            float(np.mean(batch_losses))
            if batch_losses
            else np.nan
        )

        # Validation observations are not augmented.
        val_metrics = evaluate_eegnet_loader(
            model,
            val_loader,
            device=device,
        )

        history_rows.append({
            "training_epoch": int(training_epoch),
            "sigma": float(sigma),
            "train_loss": train_loss,
            "val_loss": float(val_metrics["loss"]),
            "val_auc": float(val_metrics["auc"]),
            "val_pr_auc": float(val_metrics["pr_auc"]),
            "val_bal_acc": float(val_metrics["bal_acc"]),
        })

        if verbose and (
                training_epoch == 1
                or training_epoch % 5 == 0
                or training_epoch == cfg.eegnet.epochs
        ):
            print(
                f"epoch={training_epoch:03d} "
                f"sigma={sigma:.3f} "
                f"train_loss={train_loss:.4f} "
                f"val_loss={val_metrics['loss']:.4f} "
                f"val_auc={val_metrics['auc']:.4f}"
            )

        improved = (
                np.isfinite(val_metrics["auc"])
                and val_metrics["auc"]
                > best_val_auc + MIN_VAL_IMPROVEMENT
        )

        if improved:
            best_val_auc = float(
                val_metrics["auc"]
            )

            best_epoch = int(
                training_epoch
            )

            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model.state_dict().items()
            }

            patience_left = int(
                cfg.eegnet.early_stopping_patience
            )

        else:
            patience_left -= 1

            if patience_left <= 0:
                break

    if best_state is None:
        raise RuntimeError(
            "EEGNet produced no finite validation ROC-AUC."
        )

    model.load_state_dict(best_state)

    # Test observations are evaluated only in the frozen full stage.
    test_metrics = None

    if evaluate_test:
        test_metrics = evaluate_eegnet_loader(
            model,
            test_loader,
            device=device,
        )

    elapsed_seconds = (
            time.time() - start_time
    )

    result = {
        "sigma": float(sigma),
        "best_epoch": int(best_epoch),
        "best_val_auc": float(best_val_auc),
        "n_training_epochs_completed": int(
            len(history_rows)
        ),
        "elapsed_seconds": float(
            elapsed_seconds
        ),
        "n_inner_train": int(
            len(split.y_train)
        ),
        "n_inner_val": int(
            len(split.y_val)
        ),
        "n_outer_test": int(
            len(split.y_test)
        ),
        "train_positive_rate": float(
            split.y_train.mean()
        ),
        "val_positive_rate": float(
            split.y_val.mean()
        ),
        "test_positive_rate": float(
            split.y_test.mean()
        ),
        "normalization_mean_shape": list(
            norm_stats.mean.shape
        ),
        "normalization_std_shape": list(
            norm_stats.std.shape
        ),
        "history_df": pd.DataFrame(
            history_rows
        ),
        "test_metrics": scalar_metrics_only(
            test_metrics
        ),
    }

    del model
    del optimizer
    del train_loader
    del val_loader
    del test_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Reconstructing the final outer partitions

The notebook uses the same final experiment-family configurations as the
103-subject benchmark.

### Grouped within-subject

- trials remain in chronological groups;
- the same dataset-specific group sizes are used;
- the configured neighbouring groups are purged;
- the final valid folds are reconstructed;
- the original EEGNet training-seed convention is retained:
  `training_seed = base_seed + fold_number`.

### LOSO

- the held-out subject is used only as the outer test set;
- training subjects are concatenated with unique group identifiers;
- the inner validation split contains only complete groups from the training subjects;
- the held-out subject is never used for normalization, early stopping, or noise-scale selection.

In [ ]:
# ------------------------------------------------------------
# LOSO concatenation helper
# ------------------------------------------------------------

def concatenate_training_subjects_with_unique_groups(
        data,
        train_subjects,
):
    X_parts = []
    y_parts = []
    group_parts = []

    group_offset = 0

    for subject in train_subjects:
        X_subject = np.asarray(
            data.X_by_sub[subject],
            dtype=np.float32,
        )

        y_subject = np.asarray(
            data.y_by_sub[subject],
            dtype=np.int64,
        )

        groups_subject = np.asarray(
            data.groups_by_sub[subject],
            dtype=np.int64,
        )

        groups_unique = (
                groups_subject
                + group_offset
        )

        X_parts.append(X_subject)
        y_parts.append(y_subject)
        group_parts.append(groups_unique)

        group_offset = (
                int(groups_unique.max())
                + 1
        )

    return (
        np.concatenate(
            X_parts,
            axis=0,
        ).astype(np.float32),
        np.concatenate(
            y_parts,
            axis=0,
        ).astype(np.int64),
        np.concatenate(
            group_parts,
            axis=0,
        ).astype(np.int64),
    )


# ------------------------------------------------------------
# Dataset filter
# ------------------------------------------------------------

def dataset_is_enabled(dataset_label: str) -> bool:
    if DATASET_FILTER is None:
        return True

    return dataset_label in set(
        DATASET_FILTER
    )


# ------------------------------------------------------------
# Within-subject outer partitions
# ------------------------------------------------------------

def iter_within_subject_partitions(
        experiment_name: str,
        *,
        stage: str,
):
    cfg = get_experiment(
        experiment_name
    )

    dataset_label = dataset_label_from_cfg(
        cfg
    )

    if not dataset_is_enabled(
            dataset_label
    ):
        return

    subjects = list(
        cfg.subjects.subjects
    )

    if stage == "pilot":
        subjects = select_evenly_spaced(
            subjects,
            PILOT_SUBJECTS_PER_DATASET,
        )

        seeds = PILOT_SEEDS

    elif stage == "full":
        seeds = FULL_SEEDS

    else:
        raise ValueError(
            f"Unknown stage: {stage}"
        )

    epoching_cfg = (
        make_epoching_config_from_experiment(
            cfg
        )
    )

    crop_tmin, crop_tmax = (
        cfg.effective_crop_window()
    )

    for subject in subjects:
        print(
            f"Loading within-subject data: "
            f"{dataset_label} sub-{subject}"
        )

        (
            subject_data,
            times,
            ch_names,
            sfreq,
        ) = load_one_subject_grouped(
            dataset=cfg.dataset.dataset,
            task_name=cfg.dataset.task,
            subject=subject,
            epoching_cfg=epoching_cfg,
            chunk_size=cfg.splits.group_chunk_size,
            apply_baseline=True,
            verbose=False,
        )

        X, times_crop = crop_time_window(
            subject_data.X,
            times,
            crop_tmin,
            crop_tmax,
        )

        y = np.asarray(
            subject_data.y,
            dtype=np.int64,
        )

        groups = np.asarray(
            subject_data.groups,
            dtype=np.int64,
        )

        folds = build_purged_chunk_splits(
            y=y,
            groups=groups,
            n_splits=cfg.splits.n_splits,
            purge_neighbors=(
                cfg.splits.purge_neighbor_groups
            ),
        )

        indexed_folds = list(
            enumerate(
                folds,
                start=1,
            )
        )

        if stage == "pilot":
            indexed_folds = (
                select_evenly_spaced(
                    indexed_folds,
                    PILOT_FOLDS_PER_SUBJECT,
                )
            )

        for base_seed in seeds:
            for fold_id, (
                    train_outer_idx,
                    test_outer_idx,
            ) in indexed_folds:
                training_seed = (
                        int(base_seed)
                        + int(fold_id)
                )

                metadata = {
                    "evaluation_setting": "Within-subject",
                    "experiment_name": experiment_name,
                    "dataset": cfg.dataset.dataset,
                    "dataset_label": dataset_label,
                    "task": cfg.dataset.task,
                    "event_preset": cfg.dataset.event_preset,
                    "subject": normalize_subject_id(
                        subject,
                        dataset_label,
                    ),
                    "outer_fold": int(fold_id),
                    "base_seed": int(base_seed),
                    "training_seed": int(training_seed),
                    "n_outer_train": int(
                        len(train_outer_idx)
                    ),
                    "n_outer_test": int(
                        len(test_outer_idx)
                    ),
                    "group_chunk_size": int(
                        cfg.splits.group_chunk_size
                    ),
                    "purge_neighbor_groups": int(
                        cfg.splits.purge_neighbor_groups
                    ),
                    "n_channels": int(
                        X.shape[1]
                    ),
                    "n_times": int(
                        X.shape[2]
                    ),
                    "crop_tmin": float(
                        times_crop[0]
                    ),
                    "crop_tmax": float(
                        times_crop[-1]
                    ),
                    "sfreq": float(sfreq),
                }

                yield {
                    "cfg": cfg,
                    "metadata": metadata,
                    "X_train_outer": X[
                        train_outer_idx
                    ],
                    "y_train_outer": y[
                        train_outer_idx
                    ],
                    "groups_train_outer": groups[
                        train_outer_idx
                    ],
                    "X_test_outer": X[
                        test_outer_idx
                    ],
                    "y_test_outer": y[
                        test_outer_idx
                    ],
                }


# ------------------------------------------------------------
# LOSO outer partitions
# ------------------------------------------------------------

def iter_loso_partitions(
        experiment_name: str,
        *,
        stage: str,
):
    family = get_experiment_family(
        experiment_name
    )

    cfg = family["loso"]

    dataset_label = dataset_label_from_cfg(
        cfg
    )

    if not dataset_is_enabled(
            dataset_label
    ):
        return

    epoching_cfg = (
        make_epoching_config_from_experiment(
            cfg
        )
    )

    print(
        f"Loading complete LOSO dataset: "
        f"{dataset_label}"
    )

    data = load_subjects_grouped(
        dataset=cfg.dataset.dataset,
        task_name=cfg.dataset.task,
        subjects=cfg.subjects.subjects,
        epoching_cfg=epoching_cfg,
        chunk_size=cfg.splits.group_chunk_size,
        apply_baseline=True,
        verbose=True,
    )

    crop_tmin, crop_tmax = (
        cfg.effective_crop_window()
    )

    times_crop_reference = None

    for subject in data.subjects:
        X_crop, times_crop = crop_time_window(
            data.X_by_sub[subject],
            data.times,
            crop_tmin,
            crop_tmax,
        )

        # Replace the full -0.2 to 0.8 s array rather than constructing
        # and retaining a second complete LoadedDataset.
        data.X_by_sub[subject] = X_crop

        if times_crop_reference is None:
            times_crop_reference = times_crop.copy()
        elif not np.allclose(
                times_crop_reference,
                times_crop,
        ):
            raise ValueError(
                f"Cropped time mismatch for subject {subject}"
            )

    data.times = np.asarray(
        times_crop_reference
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    heldout_subjects = list(
        data.subjects
    )

    if stage == "pilot":
        heldout_subjects = (
            select_evenly_spaced(
                heldout_subjects,
                PILOT_SUBJECTS_PER_DATASET,
            )
        )

        seeds = PILOT_SEEDS

    elif stage == "full":
        seeds = FULL_SEEDS

    else:
        raise ValueError(
            f"Unknown stage: {stage}"
        )

    for heldout_subject in heldout_subjects:
        train_subjects = [
            subject
            for subject in data.subjects
            if subject != heldout_subject
        ]

        (
            X_train_outer,
            y_train_outer,
            groups_train_outer,
        ) = concatenate_training_subjects_with_unique_groups(
            data,
            train_subjects,
        )

        X_test_outer = np.asarray(
            data.X_by_sub[heldout_subject],
            dtype=np.float32,
        )

        y_test_outer = np.asarray(
            data.y_by_sub[heldout_subject],
            dtype=np.int64,
        )

        for base_seed in seeds:
            training_seed = int(
                base_seed
            )

            metadata = {
                "evaluation_setting": "LOSO",
                "experiment_name": experiment_name,
                "dataset": cfg.dataset.dataset,
                "dataset_label": dataset_label,
                "task": cfg.dataset.task,
                "event_preset": cfg.dataset.event_preset,
                "subject": normalize_subject_id(
                    heldout_subject,
                    dataset_label,
                ),
                "outer_fold": 1,
                "base_seed": int(base_seed),
                "training_seed": int(training_seed),
                "n_outer_train": int(
                    len(y_train_outer)
                ),
                "n_outer_test": int(
                    len(y_test_outer)
                ),
                "n_training_subjects": int(
                    len(train_subjects)
                ),
                "group_chunk_size": int(
                    cfg.splits.group_chunk_size
                ),
                "purge_neighbor_groups": int(
                    cfg.splits.purge_neighbor_groups
                ),
                "n_channels": int(
                    X_train_outer.shape[1]
                ),
                "n_times": int(
                    X_train_outer.shape[2]
                ),
                "crop_tmin": float(
                    data.times[0]
                ),
                "crop_tmax": float(
                    data.times[-1]
                ),
                "sfreq": float(
                    data.sfreq
                ),
            }

            yield {
                "cfg": cfg,
                "metadata": metadata,
                "X_train_outer": X_train_outer,
                "y_train_outer": y_train_outer,
                "groups_train_outer": groups_train_outer,
                "X_test_outer": X_test_outer,
                "y_test_outer": y_test_outer,
            }


# ------------------------------------------------------------
# Combined requested partition iterator
# ------------------------------------------------------------

def iter_requested_partitions(
        *,
        stage: str,
):
    for experiment_name in EXPERIMENT_NAMES:
        if RUN_WITHIN_SUBJECT:
            yield from iter_within_subject_partitions(
                experiment_name,
                stage=stage,
            )

        if RUN_LOSO:
            yield from iter_loso_partitions(
                experiment_name,
                stage=stage,
            )

## Stage A — Validation-only pilot

The pilot trains four conditions on each selected partition:

- \(\sigma=0\): baseline;
- \(\sigma=0.01\);
- \(\sigma=0.025\);
- \(\sigma=0.05\).

The outer test arrays are carried through the split container so that the code uses
the same data structure as the final run, but `evaluate_test=False` prevents any
outer-test metric from being calculated.

The output is checkpointed after every condition. Re-running this cell resumes
from completed JSON records.

In [ ]:
# ------------------------------------------------------------
# Run validation-only pilot
# ------------------------------------------------------------

if RUN_STAGE != "pilot":
    print(
        "RUN_STAGE is not 'pilot'. "
        "Pilot training was not started."
    )

else:
    pilot_sigmas = (
        BASELINE_SIGMA,
        *SIGMA_CANDIDATES,
    )

    completed_count = 0
    skipped_count = 0
    failed_count = 0

    for partition in iter_requested_partitions(
            stage="pilot"
    ):
        cfg = partition["cfg"]
        metadata = partition["metadata"]


        try:
            split = (
                make_inner_split_from_outer_partition(
                    X_train_outer=partition[
                        "X_train_outer"
                    ],
                    y_train_outer=partition[
                        "y_train_outer"
                    ],
                    groups_train_outer=partition[
                        "groups_train_outer"
                    ],
                    X_test_outer=partition[
                        "X_test_outer"
                    ],
                    y_test_outer=partition[
                        "y_test_outer"
                    ],
                    cfg=cfg,
                    training_seed=metadata[
                        "training_seed"
                    ],
                )
            )

            for sigma in pilot_sigmas:
                run_id = make_run_id(
                    metadata,
                    sigma,
                )

                result_path = (
                        PILOT_RUN_DIR
                        / f"{run_id}.json"
                )

                history_path = (
                        PILOT_HISTORY_DIR
                        / f"{run_id}.csv"
                )

                if (
                        result_path.exists()
                        and not OVERWRITE_PILOT
                ):
                    skipped_count += 1
                    continue

                print("-" * 90)
                print(
                    metadata[
                        "evaluation_setting"
                    ],
                    metadata["dataset_label"],
                    f"sub-{metadata['subject']}",
                    f"fold={metadata['outer_fold']}",
                    f"seed={metadata['base_seed']}",
                    f"sigma={sigma}",
                )

                result = train_eegnet_condition(
                    split=split,
                    cfg=cfg,
                    sigma=float(sigma),
                    training_seed=metadata[
                        "training_seed"
                    ],
                    device=DEVICE,
                    evaluate_test=False,
                    verbose=False,
                )

                result["history_df"].to_csv(
                    history_path,
                    index=False,
                )

                record = {
                    "status": "complete",
                    "stage": "pilot",
                    "condition": (
                        "baseline"
                        if float(sigma) == 0
                        else "gaussian_candidate"
                    ),
                    "sigma": float(sigma),
                    "best_epoch": result[
                        "best_epoch"
                    ],
                    "best_val_auc": result[
                        "best_val_auc"
                    ],
                    "n_training_epochs_completed": result[
                        "n_training_epochs_completed"
                    ],
                    "elapsed_seconds": result[
                        "elapsed_seconds"
                    ],
                    "n_inner_train": result[
                        "n_inner_train"
                    ],
                    "n_inner_val": result[
                        "n_inner_val"
                    ],
                    "n_outer_test": result[
                        "n_outer_test"
                    ],
                    "train_positive_rate": result[
                        "train_positive_rate"
                    ],
                    "val_positive_rate": result[
                        "val_positive_rate"
                    ],
                    "test_positive_rate": result[
                        "test_positive_rate"
                    ],
                    "history_path": str(
                        history_path
                    ),
                    **metadata,
                }

                save_json(
                    record,
                    result_path,
                )

                completed_count += 1

        except Exception as exc:
            failed_count += 1

            print(
                "Pilot partition failed:",
                type(exc).__name__,
                str(exc),
            )

            write_error_record(
                stage="pilot",
                metadata=metadata,
                sigma=None,
                error=exc,
            )

            if not CONTINUE_ON_ERROR:
                raise

    print("=" * 90)
    print("Pilot completed conditions:", completed_count)
    print("Pilot skipped conditions:", skipped_count)
    print("Pilot failed partitions:", failed_count)
    print("Pilot records:", PILOT_RUN_DIR)

In [ ]:
# ------------------------------------------------------------
# Pilot summary and validation-only candidate comparison
# ------------------------------------------------------------

pilot_df = load_json_records(
    PILOT_RUN_DIR
)

if pilot_df.empty:
    raise RuntimeError(
        "No pilot run records were found. "
        "Run Cell 12 with RUN_STAGE='pilot'."
    )

pilot_df = pilot_df[
    pilot_df["status"].eq("complete")
].copy()

pilot_df["sigma"] = pd.to_numeric(
    pilot_df["sigma"],
    errors="coerce",
)

pilot_df["best_val_auc"] = pd.to_numeric(
    pilot_df["best_val_auc"],
    errors="coerce",
)

pilot_df.to_csv(
    PILOT_TABLE_DIR
    / "pilot_condition_level_results.csv",
    index=False,
)


partition_key_cols = [
    "evaluation_setting",
    "dataset_label",
    "subject",
    "outer_fold",
    "base_seed",
    "training_seed",
]

expected_sigmas = {
    float(BASELINE_SIGMA),
    *[float(sigma) for sigma in SIGMA_CANDIDATES],
}

completeness_df = (
    pilot_df
    .groupby(
        partition_key_cols,
        as_index=False,
    )
    .agg(
        n_conditions=("sigma", "nunique"),
        sigma_values=(
            "sigma",
            lambda values: tuple(
                sorted(
                    set(
                        float(value)
                        for value in values
                    )
                )
            ),
        ),
    )
)

completeness_df["is_complete"] = (
    completeness_df["sigma_values"]
    .apply(set)
    .eq(expected_sigmas)
)

incomplete_df = completeness_df[
    ~completeness_df["is_complete"]
].copy()

if not incomplete_df.empty:
    print(
        "WARNING: incomplete pilot partitions:",
        len(incomplete_df),
    )
    display(incomplete_df.head(20))


pilot_pivot = pilot_df.pivot_table(
    index=partition_key_cols,
    columns="sigma",
    values="best_val_auc",
    aggfunc="first",
)

pilot_pivot.columns = [
    float(column)
    for column in pilot_pivot.columns
]

if float(BASELINE_SIGMA) not in pilot_pivot.columns:
    raise RuntimeError(
        "Pilot baseline sigma=0 was not found."
    )


pilot_summary_rows = []

for sigma in SIGMA_CANDIDATES:
    sigma = float(sigma)

    if sigma not in pilot_pivot.columns:
        continue

    pair = pilot_pivot[
        [
            float(BASELINE_SIGMA),
            sigma,
        ]
    ].dropna()

    delta = (
            pair[sigma]
            - pair[float(BASELINE_SIGMA)]
    )

    pilot_summary_rows.append({
        "sigma": sigma,
        "n_paired_partitions": int(
            len(delta)
        ),
        "baseline_mean_val_auc": float(
            pair[
                float(BASELINE_SIGMA)
            ].mean()
        ),
        "candidate_mean_val_auc": float(
            pair[sigma].mean()
        ),
        "mean_val_auc_delta": float(
            delta.mean()
        ),
        "median_val_auc_delta": float(
            delta.median()
        ),
        "val_delta_sd": float(
            delta.std(ddof=1)
        ),
        "proportion_validation_improved": float(
            np.mean(delta > 0)
        ),
    })

pilot_sigma_summary_df = (
    pd.DataFrame(
        pilot_summary_rows
    )
    .sort_values(
        [
            "mean_val_auc_delta",
            "sigma",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

pilot_sigma_summary_df.to_csv(
    PILOT_TABLE_DIR
    / "pilot_sigma_validation_summary.csv",
    index=False,
)

display(
    round_numeric_columns(
        pilot_sigma_summary_df,
        decimals=5,
    )
)

if pilot_sigma_summary_df.empty:
    raise RuntimeError(
        "No complete nonzero pilot candidate was available."
    )

recommended_sigma = float(
    pilot_sigma_summary_df.iloc[0][
        "sigma"
    ]
)

print("=" * 90)
print(
    "Validation-only recommended sigma:",
    recommended_sigma,
)
print(
    "No outer-test AUC was used in this recommendation."
)
print("=" * 90)

In [ ]:
# ------------------------------------------------------------
# Freeze one sigma before the full outer-test experiment
# ------------------------------------------------------------

# Leave as None to use the validation-only recommendation.
# A manual override must still be one of SIGMA_CANDIDATES and must
# be justified before any full test output is inspected.
MANUAL_FROZEN_SIGMA = None

# Change to True only after reviewing pilot_sigma_summary_df.
CONFIRM_FREEZE = True


chosen_sigma = (
    float(MANUAL_FROZEN_SIGMA)
    if MANUAL_FROZEN_SIGMA is not None
    else float(recommended_sigma)
)

if chosen_sigma not in {
    float(value)
    for value in SIGMA_CANDIDATES
}:
    raise ValueError(
        f"Chosen sigma {chosen_sigma} is not in "
        f"SIGMA_CANDIDATES={SIGMA_CANDIDATES}"
    )


freeze_payload = {
    "augmentation_type": (
        "additive zero-mean Gaussian noise"
    ),
    "application_stage": (
        "after train-only channel normalization"
    ),
    "application_partition": (
        "inner training only"
    ),
    "validation_augmented": False,
    "outer_test_augmented": False,
    "sigma_candidates": [
        float(value)
        for value in SIGMA_CANDIDATES
    ],
    "frozen_sigma": float(
        chosen_sigma
    ),
    "selection_metric": (
        "mean paired inner-validation ROC-AUC change "
        "relative to sigma=0"
    ),
    "selection_used_outer_test": False,
    "pilot_subjects_per_dataset": int(
        PILOT_SUBJECTS_PER_DATASET
    ),
    "pilot_folds_per_subject": int(
        PILOT_FOLDS_PER_SUBJECT
    ),
    "pilot_seeds": list(
        PILOT_SEEDS
    ),
    "pilot_summary_path": str(
        PILOT_TABLE_DIR
        / "pilot_sigma_validation_summary.csv"
    ),
    "freeze_timestamp": time.strftime(
        "%Y-%m-%d %H:%M:%S"
    ),
}


if CONFIRM_FREEZE:
    if FROZEN_CHOICE_PATH.exists():
        with FROZEN_CHOICE_PATH.open(
                "r",
                encoding="utf-8",
        ) as file:
            existing_choice = json.load(
                file
            )

        existing_sigma = float(
            existing_choice["frozen_sigma"]
        )

        if not np.isclose(
                existing_sigma,
                chosen_sigma,
        ):
            raise RuntimeError(
                "A different augmentation sigma has already been frozen.\n"
                f"Existing sigma: {existing_sigma}\n"
                f"Requested sigma: {chosen_sigma}\n"
                "Do not change the frozen choice after examining full test results."
            )

    save_json(
        freeze_payload,
        FROZEN_CHOICE_PATH,
    )

    print(
        "Frozen augmentation choice saved:",
        FROZEN_CHOICE_PATH,
    )
    print(
        "Frozen sigma:",
        chosen_sigma,
    )

else:
    print(
        "Sigma has NOT been frozen."
    )
    print(
        "Recommended sigma:",
        recommended_sigma,
    )
    print(
        "Set CONFIRM_FREEZE=True and rerun this cell "
        "before changing RUN_STAGE to 'full'."
    )

## Stage B — Full confirmatory ablation

Before running the full analysis:

1. confirm that `frozen_augmentation_choice.json` exists;
2. change `RUN_STAGE` in Cell 2 to `"full"`;
3. restart the kernel and run the notebook from Cell 1;
4. do not modify the sigma candidate set or frozen choice after inspecting
   full outer-test results.

The full run compares only:

- baseline EEGNet, \(\sigma=0\);
- Gaussian-noise EEGNet using the frozen \(\sigma\).

Every result is saved individually, so the experiment can be resumed safely.

When using the GTX 960M, running one dataset at a time is sensible. For example:

```python
DATASET_FILTER = {"ds003061"}

In [ ]:
# Cell 16 — Code: run full baseline-versus-augmentation experiment

# ------------------------------------------------------------
# Run full confirmatory ablation
# ------------------------------------------------------------

if RUN_STAGE != "full":
    print(
        "RUN_STAGE is not 'full'. "
        "Full test evaluation was not started."
    )

else:
    if not FROZEN_CHOICE_PATH.exists():
        raise FileNotFoundError(
            "The augmentation choice has not been frozen.\n"
            f"Missing file: {FROZEN_CHOICE_PATH}\n"
            "Complete Cells 12–14 first."
        )

    with FROZEN_CHOICE_PATH.open(
            "r",
            encoding="utf-8",
    ) as file:
        frozen_choice = json.load(
            file
        )

    frozen_sigma = float(
        frozen_choice["frozen_sigma"]
    )

    # The original seed-specific final EEGNet results are reused as
    # the baseline. Only the augmentation condition is retrained.
    full_conditions = [
        (
            "gaussian_noise",
            frozen_sigma,
        ),
    ]

    completed_count = 0
    skipped_count = 0
    failed_count = 0

    print("=" * 90)
    print("Frozen sigma:", frozen_sigma)
    print(
        "Full conditions:",
        full_conditions,
    )
    print("=" * 90)

    split = None

    for partition in iter_requested_partitions(
            stage="full"
    ):
        cfg = partition["cfg"]
        metadata = partition["metadata"]

        try:
            split = (
                make_inner_split_from_outer_partition(
                    X_train_outer=partition[
                        "X_train_outer"
                    ],
                    y_train_outer=partition[
                        "y_train_outer"
                    ],
                    groups_train_outer=partition[
                        "groups_train_outer"
                    ],
                    X_test_outer=partition[
                        "X_test_outer"
                    ],
                    y_test_outer=partition[
                        "y_test_outer"
                    ],
                    cfg=cfg,
                    training_seed=metadata[
                        "training_seed"
                    ],
                )
            )

            for condition, sigma in full_conditions:
                run_id = make_run_id(
                    metadata,
                    sigma,
                )

                result_path = (
                        FULL_RUN_DIR
                        / f"{run_id}.json"
                )

                history_path = (
                        FULL_HISTORY_DIR
                        / f"{run_id}.csv"
                )

                if (
                        result_path.exists()
                        and not OVERWRITE_FULL
                ):
                    skipped_count += 1
                    continue

                print("-" * 90)
                print(
                    metadata[
                        "evaluation_setting"
                    ],
                    metadata[
                        "dataset_label"
                    ],
                    f"sub-{metadata['subject']}",
                    f"fold={metadata['outer_fold']}",
                    f"base_seed={metadata['base_seed']}",
                    f"training_seed={metadata['training_seed']}",
                    f"condition={condition}",
                    f"sigma={sigma}",
                )

                result = train_eegnet_condition(
                    split=split,
                    cfg=cfg,
                    sigma=float(sigma),
                    training_seed=metadata[
                        "training_seed"
                    ],
                    device=DEVICE,
                    evaluate_test=True,
                    verbose=False,
                )

                result["history_df"].to_csv(
                    history_path,
                    index=False,
                )

                test_metrics = result[
                    "test_metrics"
                ]

                record = {
                    "status": "complete",
                    "stage": "full",
                    "condition": condition,
                    "sigma": float(sigma),
                    "best_epoch": result[
                        "best_epoch"
                    ],
                    "best_val_auc": result[
                        "best_val_auc"
                    ],
                    "n_training_epochs_completed": result[
                        "n_training_epochs_completed"
                    ],
                    "elapsed_seconds": result[
                        "elapsed_seconds"
                    ],
                    "n_inner_train": result[
                        "n_inner_train"
                    ],
                    "n_inner_val": result[
                        "n_inner_val"
                    ],
                    "n_outer_test": result[
                        "n_outer_test"
                    ],
                    "train_positive_rate": result[
                        "train_positive_rate"
                    ],
                    "val_positive_rate": result[
                        "val_positive_rate"
                    ],
                    "test_positive_rate": result[
                        "test_positive_rate"
                    ],
                    "test_auc": test_metrics[
                        "auc"
                    ],
                    "test_pr_auc": test_metrics[
                        "pr_auc"
                    ],
                    "test_accuracy": test_metrics[
                        "acc"
                    ],
                    "test_balanced_accuracy": test_metrics[
                        "bal_acc"
                    ],
                    "test_f1": test_metrics[
                        "f1"
                    ],
                    "test_tn": test_metrics[
                        "tn"
                    ],
                    "test_fp": test_metrics[
                        "fp"
                    ],
                    "test_fn": test_metrics[
                        "fn"
                    ],
                    "test_tp": test_metrics[
                        "tp"
                    ],
                    "history_path": str(
                        history_path
                    ),
                    "frozen_choice_path": str(
                        FROZEN_CHOICE_PATH
                    ),
                    **metadata,
                }

                save_json(
                    record,
                    result_path,
                )

                completed_count += 1

        except Exception as exc:
            failed_count += 1

            print(
                "Full partition failed:",
                type(exc).__name__,
                str(exc),
            )

            write_error_record(
                stage="full",
                metadata=metadata,
                sigma=None,
                error=exc,
            )

            if not CONTINUE_ON_ERROR:
                raise

        finally:
            # Remove the copied inner train/validation/test arrays.
            if split is not None:
                del split

            # Prevent the notebook loop variable from retaining the previous
            # dataset's large outer arrays while the next dataset is loaded.
            if isinstance(partition, dict):
                for key in (
                        "X_train_outer",
                        "y_train_outer",
                        "groups_train_outer",
                        "X_test_outer",
                        "y_test_outer",
                ):
                    if key in partition:
                        partition[key] = None

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    print("=" * 90)
    print("Full completed conditions:", completed_count)
    print("Full skipped conditions:", skipped_count)
    print("Full failed partitions:", failed_count)
    print("Full records:", FULL_RUN_DIR)

## Load the exact original seed-specific LOSO EEGNet baselines

In [ ]:
# ------------------------------------------------------------
# Load exact original seed-specific LOSO EEGNet baseline results
# ------------------------------------------------------------

def load_original_loso_eegnet_baselines(
        experiment_names,
        seeds=(42, 43, 44),
) -> pd.DataFrame:
    baseline_records = []

    for experiment_name in experiment_names:
        family = get_experiment_family(
            experiment_name
        )

        cfg_loso = family["loso"]

        dataset_label = dataset_label_from_cfg(
            cfg_loso
        )

        for seed in seeds:
            fold_results_path = (
                    cfg_loso.result_dir(seed=seed)
                    / "tables"
                    / "folds"
                    / "loso_fold_results.csv"
            )

            if not fold_results_path.exists():
                raise FileNotFoundError(
                    "Original LOSO fold results were not found:\n"
                    f"{fold_results_path}"
                )

            fold_df = pd.read_csv(
                fold_results_path
            )

            if "model" not in fold_df.columns:
                raise ValueError(
                    f"'model' column missing from {fold_results_path}"
                )

            eegnet_df = fold_df[
                fold_df["model"]
                .astype(str)
                .str.lower()
                .eq("eegnet")
            ].copy()

            if eegnet_df.empty:
                raise RuntimeError(
                    "No EEGNet baseline rows found in:\n"
                    f"{fold_results_path}"
                )

            # Build a subject-to-history map so the original number
            # of completed training epochs can also be recovered.
            history_dir = (
                    cfg_loso.result_dir(seed=seed)
                    / "histories"
                    / "eegnet"
            )

            history_map = {}

            if history_dir.exists():
                for history_path in history_dir.glob(
                        "eegnet_history_heldout-subject-*.csv"
                ):
                    subject_token = (
                        history_path.stem
                        .replace(
                            "eegnet_history_heldout-subject-",
                            "",
                        )
                    )

                    normalized_subject = normalize_subject_id(
                        subject_token,
                        dataset_label,
                    )

                    history_map[
                        normalized_subject
                    ] = history_path

            for _, row in eegnet_df.iterrows():
                original_subject = row.get(
                    "heldout_subject",
                    row.get("subject"),
                )

                subject = normalize_subject_id(
                    original_subject,
                    dataset_label,
                )

                history_path = history_map.get(
                    subject
                )

                if (
                        history_path is not None
                        and history_path.exists()
                ):
                    history_df = pd.read_csv(
                        history_path
                    )

                    n_training_epochs_completed = int(
                        len(history_df)
                    )

                    history_path_string = str(
                        history_path
                    )

                else:
                    n_training_epochs_completed = np.nan
                    history_path_string = None

                baseline_records.append({
                    "status": "complete",
                    "stage": "full",
                    "condition": "baseline",
                    "sigma": 0.0,

                    "evaluation_setting": "LOSO",
                    "experiment_name": experiment_name,
                    "dataset": cfg_loso.dataset.dataset,
                    "dataset_label": dataset_label,
                    "task": cfg_loso.dataset.task,
                    "event_preset": cfg_loso.dataset.event_preset,

                    "subject": subject,
                    "outer_fold": int(
                        row.get("fold", 1)
                    ),
                    "base_seed": int(seed),
                    "training_seed": int(seed),

                    "n_outer_train": int(
                        row["n_train"]
                    ),
                    "n_outer_test": int(
                        row["n_test"]
                    ),

                    "train_positive_rate": float(
                        row["train_pos_rate"]
                    ),
                    "test_positive_rate": float(
                        row["test_pos_rate"]
                    ),

                    "best_epoch": float(
                        row["best_epoch"]
                    ),
                    "best_val_auc": float(
                        row["best_val_auc"]
                    ),
                    "n_training_epochs_completed": (
                        n_training_epochs_completed
                    ),

                    "test_auc": float(
                        row["auc"]
                    ),
                    "test_pr_auc": float(
                        row["pr_auc"]
                    ),
                    "test_accuracy": float(
                        row["acc"]
                    ),
                    "test_balanced_accuracy": float(
                        row["bal_acc"]
                    ),
                    "test_f1": float(
                        row["f1"]
                    ),

                    "test_tn": int(
                        row["tn"]
                    ),
                    "test_fp": int(
                        row["fp"]
                    ),
                    "test_fn": int(
                        row["fn"]
                    ),
                    "test_tp": int(
                        row["tp"]
                    ),

                    "history_path": (
                        history_path_string
                    ),
                    "baseline_source_path": str(
                        fold_results_path
                    ),
                })

    baseline_df = pd.DataFrame(
        baseline_records
    )

    duplicate_mask = baseline_df.duplicated(
        subset=[
            "evaluation_setting",
            "dataset_label",
            "subject",
            "base_seed",
        ],
        keep=False,
    )

    if duplicate_mask.any():
        duplicate_rows = baseline_df.loc[
            duplicate_mask,
            [
                "dataset_label",
                "subject",
                "base_seed",
            ],
        ]

        display(duplicate_rows)

        raise RuntimeError(
            "Duplicate original baseline rows were found."
        )

    expected_rows = (
            103
            * len(tuple(seeds))
    )

    print("=" * 90)
    print(
        "Original seed-specific EEGNet baseline rows:",
        len(baseline_df),
    )
    print(
        "Expected baseline rows:",
        expected_rows,
    )
    print("=" * 90)

    if len(baseline_df) != expected_rows:
        raise RuntimeError(
            "The original baseline table is incomplete. "
            f"Expected {expected_rows} rows but found "
            f"{len(baseline_df)}."
        )

    return baseline_df


original_baseline_df = (
    load_original_loso_eegnet_baselines(
        EXPERIMENT_NAMES,
        seeds=FULL_SEEDS,
    )
)

original_baseline_df.to_csv(
    FULL_TABLE_DIR
    / "original_loso_eegnet_seed_baselines.csv",
    index=False,
)

display(
    original_baseline_df.groupby(
        [
            "dataset_label",
            "base_seed",
        ],
        as_index=False,
    ).agg(
        n_subjects=("subject", "nunique"),
        mean_auc=("test_auc", "mean"),
        mean_val_auc=("best_val_auc", "mean"),
    )
)

In [ ]:
# ------------------------------------------------------------
# Consolidate condition-level and paired outer-partition results
# ------------------------------------------------------------

# ------------------------------------------------------------
# Combine imported original baselines with new augmentation runs
# ------------------------------------------------------------

augmented_run_df = load_json_records(
    FULL_RUN_DIR
)

if augmented_run_df.empty:
    raise RuntimeError(
        "No completed augmented full-run records were found."
    )

augmented_run_df = augmented_run_df[
    augmented_run_df["status"].eq(
        "complete"
    )
].copy()

# Only newly trained augmented runs belong in this table.
augmented_run_df = augmented_run_df[
    augmented_run_df["condition"].eq(
        "gaussian_noise"
    )
].copy()

numeric_cols = [
    "sigma",
    "best_epoch",
    "best_val_auc",
    "n_training_epochs_completed",
    "elapsed_seconds",
    "test_auc",
    "test_pr_auc",
    "test_balanced_accuracy",
    "test_f1",
]

for column in numeric_cols:
    if column in augmented_run_df.columns:
        augmented_run_df[column] = pd.to_numeric(
            augmented_run_df[column],
            errors="coerce",
        )

full_df = pd.concat(
    [
        original_baseline_df,
        augmented_run_df,
    ],
    ignore_index=True,
    sort=False,
)

full_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_condition_level_results.csv",
    index=False,
)

pair_key_cols = [
    "evaluation_setting",
    "experiment_name",
    "dataset",
    "dataset_label",
    "subject",
    "outer_fold",
    "base_seed",
    "training_seed",
]

baseline_run_df = (
    full_df[
        full_df["condition"].eq(
            "baseline"
        )
    ]
    [
        pair_key_cols
        + [
            "sigma",
            "best_epoch",
            "best_val_auc",
            "n_training_epochs_completed",
            "test_auc",
            "test_pr_auc",
            "test_balanced_accuracy",
            "test_f1",
        ]
        ]
    .rename(columns={
        "sigma": "baseline_sigma",
        "best_epoch": "baseline_best_epoch",
        "best_val_auc": "baseline_val_auc",
        "n_training_epochs_completed": (
            "baseline_epochs_completed"
        ),
        "test_auc": "baseline_test_auc",
        "test_pr_auc": "baseline_test_pr_auc",
        "test_balanced_accuracy": (
            "baseline_test_bal_acc"
        ),
        "test_f1": "baseline_test_f1",
    })
)

augmented_run_df = (
    full_df[
        full_df["condition"].eq(
            "gaussian_noise"
        )
    ]
    [
        pair_key_cols
        + [
            "sigma",
            "best_epoch",
            "best_val_auc",
            "n_training_epochs_completed",
            "test_auc",
            "test_pr_auc",
            "test_balanced_accuracy",
            "test_f1",
        ]
        ]
    .rename(columns={
        "sigma": "augmented_sigma",
        "best_epoch": "augmented_best_epoch",
        "best_val_auc": "augmented_val_auc",
        "n_training_epochs_completed": (
            "augmented_epochs_completed"
        ),
        "test_auc": "augmented_test_auc",
        "test_pr_auc": "augmented_test_pr_auc",
        "test_balanced_accuracy": (
            "augmented_test_bal_acc"
        ),
        "test_f1": "augmented_test_f1",
    })
)

outer_comparison_df = baseline_run_df.merge(
    augmented_run_df,
    on=pair_key_cols,
    how="inner",
    validate="one_to_one",
)

outer_comparison_df[
    "delta_auc_augmented_minus_baseline"
] = (
        outer_comparison_df[
            "augmented_test_auc"
        ]
        - outer_comparison_df[
            "baseline_test_auc"
        ]
)

outer_comparison_df[
    "delta_val_auc_augmented_minus_baseline"
] = (
        outer_comparison_df[
            "augmented_val_auc"
        ]
        - outer_comparison_df[
            "baseline_val_auc"
        ]
)

outer_comparison_df[
    "baseline_validation_test_gap"
] = (
        outer_comparison_df[
            "baseline_val_auc"
        ]
        - outer_comparison_df[
            "baseline_test_auc"
        ]
)

outer_comparison_df[
    "augmented_validation_test_gap"
] = (
        outer_comparison_df[
            "augmented_val_auc"
        ]
        - outer_comparison_df[
            "augmented_test_auc"
        ]
)

outer_comparison_df[
    "delta_gap_augmented_minus_baseline"
] = (
        outer_comparison_df[
            "augmented_validation_test_gap"
        ]
        - outer_comparison_df[
            "baseline_validation_test_gap"
        ]
)

outer_comparison_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_outer_partition_comparisons.csv",
    index=False,
)


count_table = (
    full_df
    .groupby(
        [
            "evaluation_setting",
            "dataset_label",
            "condition",
        ],
        as_index=False,
    )
    .agg(
        n_runs=("test_auc", "size"),
        n_subjects=("subject", "nunique"),
        n_base_seeds=("base_seed", "nunique"),
    )
)

display(count_table)

EXPECTED_SEED_LEVEL_PAIRS = 309
EXPECTED_CONDITION_LEVEL_ROWS = 618
EXPECTED_SUBJECTS = 103

print("=" * 90)
print(
    "Condition-level rows:",
    len(full_df),
)
print(
    "Complete seed-level baseline/augmentation pairs:",
    len(outer_comparison_df),
)
print(
    "Expected seed-level pairs:",
    EXPECTED_SEED_LEVEL_PAIRS,
)
print(
    "Expected condition-level rows:",
    EXPECTED_CONDITION_LEVEL_ROWS,
)
print("=" * 90)

if len(outer_comparison_df) != EXPECTED_SEED_LEVEL_PAIRS:
    raise RuntimeError(
        "The full paired augmentation experiment is incomplete. "
        f"Expected {EXPECTED_SEED_LEVEL_PAIRS} pairs but found "
        f"{len(outer_comparison_df)}."
    )

if len(full_df) != EXPECTED_CONDITION_LEVEL_ROWS:
    raise RuntimeError(
        "The condition-level table is incomplete. "
        f"Expected {EXPECTED_CONDITION_LEVEL_ROWS} rows but found "
        f"{len(full_df)}."
    )

n_subjects = (
    outer_comparison_df[
        [
            "dataset_label",
            "subject",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Unique held-out subject-recordings:",
    n_subjects,
)

if n_subjects != EXPECTED_SUBJECTS:
    raise RuntimeError(
        f"Expected {EXPECTED_SUBJECTS} subject-recordings "
        f"but found {n_subjects}."
    )

In [ ]:
# ------------------------------------------------------------
# Aggregate folds and seeds before statistical inference
# ------------------------------------------------------------

subject_condition_df = (
    full_df
    .groupby(
        [
            "evaluation_setting",
            "dataset_label",
            "subject",
            "condition",
        ],
        as_index=False,
    )
    .agg(
        n_outer_runs=("test_auc", "size"),
        mean_test_auc=("test_auc", "mean"),
        sd_test_auc=("test_auc", "std"),
        mean_val_auc=("best_val_auc", "mean"),
        mean_best_epoch=("best_epoch", "mean"),
        mean_epochs_completed=(
            "n_training_epochs_completed",
            "mean",
        ),
    )
)

subject_condition_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_subject_condition_summary.csv",
    index=False,
)


subject_baseline_df = (
    subject_condition_df[
        subject_condition_df[
            "condition"
        ].eq("baseline")
    ]
    .rename(columns={
        "n_outer_runs": "baseline_n_outer_runs",
        "mean_test_auc": "baseline_auc",
        "sd_test_auc": "baseline_run_auc_sd",
        "mean_val_auc": "baseline_mean_val_auc",
        "mean_best_epoch": "baseline_mean_best_epoch",
        "mean_epochs_completed": (
            "baseline_mean_epochs_completed"
        ),
    })
    .drop(columns=["condition"])
)

subject_augmented_df = (
    subject_condition_df[
        subject_condition_df[
            "condition"
        ].eq("gaussian_noise")
    ]
    .rename(columns={
        "n_outer_runs": "augmented_n_outer_runs",
        "mean_test_auc": "augmented_auc",
        "sd_test_auc": "augmented_run_auc_sd",
        "mean_val_auc": "augmented_mean_val_auc",
        "mean_best_epoch": "augmented_mean_best_epoch",
        "mean_epochs_completed": (
            "augmented_mean_epochs_completed"
        ),
    })
    .drop(columns=["condition"])
)

subject_comparison_df = (
    subject_baseline_df
    .merge(
        subject_augmented_df,
        on=[
            "evaluation_setting",
            "dataset_label",
            "subject",
        ],
        how="inner",
        validate="one_to_one",
    )
)

subject_comparison_df[
    "delta_auc_augmented_minus_baseline"
] = (
        subject_comparison_df[
            "augmented_auc"
        ]
        - subject_comparison_df[
            "baseline_auc"
        ]
)

subject_comparison_df[
    "delta_val_auc_augmented_minus_baseline"
] = (
        subject_comparison_df[
            "augmented_mean_val_auc"
        ]
        - subject_comparison_df[
            "baseline_mean_val_auc"
        ]
)

subject_comparison_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_subject_level_comparison.csv",
    index=False,
)


dataset_summary_df = (
    subject_comparison_df
    .groupby(
        [
            "evaluation_setting",
            "dataset_label",
        ],
        as_index=False,
    )
    .agg(
        n_subjects=("subject", "size"),
        baseline_mean_auc=("baseline_auc", "mean"),
        baseline_subject_sd=("baseline_auc", "std"),
        augmented_mean_auc=("augmented_auc", "mean"),
        augmented_subject_sd=("augmented_auc", "std"),
        mean_delta=(
            "delta_auc_augmented_minus_baseline",
            "mean",
        ),
        median_delta=(
            "delta_auc_augmented_minus_baseline",
            "median",
        ),
        delta_subject_sd=(
            "delta_auc_augmented_minus_baseline",
            "std",
        ),
        proportion_improved=(
            "delta_auc_augmented_minus_baseline",
            lambda values: float(
                np.mean(
                    np.asarray(values) > 0
                )
            ),
        ),
    )
)

dataset_summary_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_dataset_summary.csv",
    index=False,
)

display(
    round_numeric_columns(
        dataset_summary_df,
        decimals=4,
    )
)

print(
    "Subject-level rows:",
    len(subject_comparison_df),
)

In [ ]:
# ------------------------------------------------------------
# Overall subject-level augmentation tests
# ------------------------------------------------------------

overall_rows = []

for evaluation_setting, group_df in (
        subject_comparison_df
                .groupby("evaluation_setting")
):
    delta = group_df[
        "delta_auc_augmented_minus_baseline"
    ].to_numpy(dtype=float)

    statistic, p_value, n_subjects = (
        safe_wilcoxon(delta)
    )

    ci_low, ci_high = bootstrap_mean_ci(
        delta,
        seed=42,
    )

    overall_rows.append({
        "evaluation_setting": evaluation_setting,
        "n_subjects": int(n_subjects),
        "baseline_mean_auc": float(
            group_df["baseline_auc"].mean()
        ),
        "baseline_subject_sd": float(
            group_df["baseline_auc"].std(ddof=1)
        ),
        "augmented_mean_auc": float(
            group_df["augmented_auc"].mean()
        ),
        "augmented_subject_sd": float(
            group_df["augmented_auc"].std(ddof=1)
        ),
        "mean_delta": float(
            np.mean(delta)
        ),
        "delta_subject_sd": float(
            np.std(delta, ddof=1)
        ),
        "median_delta": float(
            np.median(delta)
        ),
        "bootstrap_mean_delta_95ci_low": ci_low,
        "bootstrap_mean_delta_95ci_high": ci_high,
        "wilcoxon_statistic": statistic,
        "wilcoxon_p_value": p_value,
        "rank_biserial": matched_rank_biserial(
            delta
        ),
        "proportion_improved": float(
            np.mean(delta > 0)
        ),
        "proportion_unchanged": float(
            np.mean(delta == 0)
        ),
        "proportion_decreased": float(
            np.mean(delta < 0)
        ),
    })

overall_statistics_df = pd.DataFrame(
    overall_rows
)

overall_statistics_df[
    "wilcoxon_p_holm"
] = holm_adjust(
    overall_statistics_df[
        "wilcoxon_p_value"
    ].to_numpy()
)

overall_statistics_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_overall_subject_statistics.csv",
    index=False,
)

display(
    round_numeric_columns(
        overall_statistics_df,
        decimals=5,
    )
)

In [ ]:
# ------------------------------------------------------------
# Load and merge retrospective Subject Quality Index
# ------------------------------------------------------------

if not QUALITY_INDEX_PATH.exists():
    raise FileNotFoundError(
        "Subject Quality Index table was not found.\n"
        f"Expected path: {QUALITY_INDEX_PATH}\n"
        "Run Notebook 24 before continuing."
    )

quality_df = pd.read_csv(
    QUALITY_INDEX_PATH
)

required_quality_cols = {
    "dataset_label",
    "subject",
    "subject_quality_index",
}

missing_quality_cols = (
        required_quality_cols
        - set(quality_df.columns)
)

if missing_quality_cols:
    raise ValueError(
        "Subject Quality Index table is missing columns: "
        f"{sorted(missing_quality_cols)}"
    )

quality_df = quality_df[
    [
        "dataset_label",
        "subject",
        "subject_quality_index",
    ]
].copy()

quality_df["dataset_label"] = (
    quality_df["dataset_label"]
    .replace({
        "ds006480": "ds006480_pooled",
        "ds006480 pooled": "ds006480_pooled",
    })
)

quality_df["subject"] = [
    normalize_subject_id(
        subject,
        dataset_label,
    )
    for subject, dataset_label
    in zip(
        quality_df["subject"],
        quality_df["dataset_label"],
    )
]

quality_df[
    "subject_quality_index"
] = pd.to_numeric(
    quality_df[
        "subject_quality_index"
    ],
    errors="coerce",
)


def assign_quality_groups(
        dataset_quality_df: pd.DataFrame,
) -> pd.DataFrame:
    dataset_quality_df = (
        dataset_quality_df.copy()
    )

    q1 = dataset_quality_df[
        "subject_quality_index"
    ].quantile(0.25)

    q3 = dataset_quality_df[
        "subject_quality_index"
    ].quantile(0.75)

    labels = np.full(
        len(dataset_quality_df),
        "middle",
        dtype=object,
    )

    quality_values = dataset_quality_df[
        "subject_quality_index"
    ].to_numpy()

    labels[
        quality_values <= q1
        ] = "hard"

    labels[
        quality_values >= q3
        ] = "easy"

    dataset_quality_df[
        "quality_group"
    ] = labels

    dataset_quality_df[
        "quality_q1"
    ] = float(q1)

    dataset_quality_df[
        "quality_q3"
    ] = float(q3)

    return dataset_quality_df


quality_group_df = (
    quality_df
    .groupby(
        "dataset_label",
        group_keys=False,
    )
    .apply(assign_quality_groups)
    .reset_index(drop=True)
)

subject_quality_comparison_df = (
    subject_comparison_df
    .merge(
        quality_group_df,
        on=[
            "dataset_label",
            "subject",
        ],
        how="left",
        validate="many_to_one",
    )
)

merge_check_df = (
    subject_quality_comparison_df
    .assign(
        has_quality=lambda data: (
            data[
                "subject_quality_index"
            ].notna()
        )
    )
    .groupby(
        "evaluation_setting",
        as_index=False,
    )
    .agg(
        n_subjects=("subject", "size"),
        n_with_quality=("has_quality", "sum"),
    )
)

display(merge_check_df)

subject_quality_comparison_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_subject_level_with_quality.csv",
    index=False,
)

display(
    round_numeric_columns(
        subject_quality_comparison_df.head(20),
        decimals=4,
    )
)

In [ ]:
# ------------------------------------------------------------
# Easy/hard summaries
# ------------------------------------------------------------

quartile_df = (
    subject_quality_comparison_df
    .query(
        "quality_group in ['hard', 'easy']"
    )
    .copy()
)

easy_hard_summary_df = (
    quartile_df
    .groupby(
        [
            "evaluation_setting",
            "quality_group",
        ],
        as_index=False,
    )
    .agg(
        n_subjects=("subject", "size"),
        baseline_mean_auc=("baseline_auc", "mean"),
        baseline_subject_sd=("baseline_auc", "std"),
        augmented_mean_auc=("augmented_auc", "mean"),
        augmented_subject_sd=("augmented_auc", "std"),
        mean_delta=(
            "delta_auc_augmented_minus_baseline",
            "mean",
        ),
        delta_subject_sd=(
            "delta_auc_augmented_minus_baseline",
            "std",
        ),
        median_delta=(
            "delta_auc_augmented_minus_baseline",
            "median",
        ),
        proportion_improved=(
            "delta_auc_augmented_minus_baseline",
            lambda values: float(
                np.mean(
                    np.asarray(values) > 0
                )
            ),
        ),
    )
)

easy_hard_summary_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_easy_hard_summary.csv",
    index=False,
)


# ------------------------------------------------------------
# Paired augmentation test within each quality group
# ------------------------------------------------------------

quality_group_test_rows = []

for (
        evaluation_setting,
        quality_group,
), group_df in quartile_df.groupby(
    [
        "evaluation_setting",
        "quality_group",
    ]
):
    delta = group_df[
        "delta_auc_augmented_minus_baseline"
    ].to_numpy(dtype=float)

    statistic, p_value, n_subjects = (
        safe_wilcoxon(delta)
    )

    ci_low, ci_high = bootstrap_mean_ci(
        delta,
        seed=42,
    )

    quality_group_test_rows.append({
        "evaluation_setting": evaluation_setting,
        "quality_group": quality_group,
        "n_subjects": n_subjects,
        "mean_delta": float(
            np.mean(delta)
        ),
        "median_delta": float(
            np.median(delta)
        ),
        "bootstrap_mean_delta_95ci_low": ci_low,
        "bootstrap_mean_delta_95ci_high": ci_high,
        "wilcoxon_statistic": statistic,
        "wilcoxon_p_value": p_value,
        "rank_biserial": matched_rank_biserial(
            delta
        ),
        "proportion_improved": float(
            np.mean(delta > 0)
        ),
    })

quality_group_tests_df = pd.DataFrame(
    quality_group_test_rows
)

quality_group_tests_df[
    "wilcoxon_p_holm"
] = holm_adjust(
    quality_group_tests_df[
        "wilcoxon_p_value"
    ].to_numpy()
)

quality_group_tests_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_quality_group_paired_tests.csv",
    index=False,
)


# ------------------------------------------------------------
# Hard-versus-easy difference in augmentation gain
# ------------------------------------------------------------

hard_easy_difference_rows = []

for evaluation_setting, setting_df in (
        quartile_df
                .groupby("evaluation_setting")
):
    hard_delta = setting_df.loc[
        setting_df[
            "quality_group"
        ].eq("hard"),
        "delta_auc_augmented_minus_baseline",
    ].dropna().to_numpy()

    easy_delta = setting_df.loc[
        setting_df[
            "quality_group"
        ].eq("easy"),
        "delta_auc_augmented_minus_baseline",
    ].dropna().to_numpy()

    if len(hard_delta) == 0 or len(easy_delta) == 0:
        statistic = np.nan
        p_value = np.nan
    else:
        statistic, p_value = mannwhitneyu(
            hard_delta,
            easy_delta,
            alternative="two-sided",
        )

    hard_easy_difference_rows.append({
        "evaluation_setting": evaluation_setting,
        "n_hard": int(len(hard_delta)),
        "n_easy": int(len(easy_delta)),
        "hard_mean_delta": float(
            np.mean(hard_delta)
        ),
        "easy_mean_delta": float(
            np.mean(easy_delta)
        ),
        "hard_minus_easy_mean_delta": float(
            np.mean(hard_delta)
            - np.mean(easy_delta)
        ),
        "mannwhitney_u": float(
            statistic
        ),
        "mannwhitney_p_value": float(
            p_value
        ),
    })

hard_easy_difference_df = pd.DataFrame(
    hard_easy_difference_rows
)

hard_easy_difference_df[
    "mannwhitney_p_holm"
] = holm_adjust(
    hard_easy_difference_df[
        "mannwhitney_p_value"
    ].to_numpy()
)

hard_easy_difference_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_hard_vs_easy_tests.csv",
    index=False,
)


# ------------------------------------------------------------
# Continuous quality-versus-augmentation-gain correlation
# ------------------------------------------------------------

quality_correlation_rows = []

for evaluation_setting, setting_df in (
        subject_quality_comparison_df
                .groupby("evaluation_setting")
):
    rho, p_value, n_subjects = safe_spearman(
        setting_df[
            "subject_quality_index"
        ],
        setting_df[
            "delta_auc_augmented_minus_baseline"
        ],
    )

    quality_correlation_rows.append({
        "evaluation_setting": evaluation_setting,
        "n_subjects": n_subjects,
        "spearman_rho_quality_vs_augmentation_gain": rho,
        "spearman_p_value": p_value,
    })

quality_correlation_df = pd.DataFrame(
    quality_correlation_rows
)

quality_correlation_df[
    "spearman_p_holm"
] = holm_adjust(
    quality_correlation_df[
        "spearman_p_value"
    ].to_numpy()
)

quality_correlation_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_quality_gain_correlations.csv",
    index=False,
)


print("Easy/hard descriptive summary")
display(
    round_numeric_columns(
        easy_hard_summary_df,
        decimals=5,
    )
)

print("\nPaired augmentation tests within groups")
display(
    round_numeric_columns(
        quality_group_tests_df,
        decimals=5,
    )
)

print("\nHard-versus-easy gain comparisons")
display(
    round_numeric_columns(
        hard_easy_difference_df,
        decimals=5,
    )
)

print("\nContinuous quality-versus-gain correlations")
display(
    round_numeric_columns(
        quality_correlation_df,
        decimals=5,
    )
)

In [ ]:
# ------------------------------------------------------------
# Training, validation, test, and generalization-gap diagnostics
# ------------------------------------------------------------

diagnostic_df = full_df.copy()

diagnostic_df[
    "validation_test_gap"
] = (
        diagnostic_df[
            "best_val_auc"
        ]
        - diagnostic_df[
            "test_auc"
        ]
)

diagnostic_summary_df = (
    diagnostic_df
    .groupby(
        [
            "evaluation_setting",
            "condition",
        ],
        as_index=False,
    )
    .agg(
        n_runs=("test_auc", "size"),
        mean_best_epoch=("best_epoch", "mean"),
        median_best_epoch=("best_epoch", "median"),
        mean_epochs_completed=(
            "n_training_epochs_completed",
            "mean",
        ),
        median_epochs_completed=(
            "n_training_epochs_completed",
            "median",
        ),
        proportion_stopped_before_max=(
            "n_training_epochs_completed",
            lambda values: float(
                np.mean(
                    np.asarray(values)
                    < 60
                )
            ),
        ),
        mean_validation_auc=("best_val_auc", "mean"),
        mean_test_auc=("test_auc", "mean"),
        mean_validation_test_gap=(
            "validation_test_gap",
            "mean",
        ),
        sd_validation_test_gap=(
            "validation_test_gap",
            "std",
        ),
    )
)

diagnostic_summary_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_training_diagnostic_summary.csv",
    index=False,
)


gap_test_rows = []

for evaluation_setting, setting_df in (
        outer_comparison_df
                .groupby("evaluation_setting")
):
    gap_delta = setting_df[
        "delta_gap_augmented_minus_baseline"
    ].to_numpy(dtype=float)

    statistic, p_value, n_runs = (
        safe_wilcoxon(gap_delta)
    )

    gap_test_rows.append({
        "evaluation_setting": evaluation_setting,
        "n_paired_outer_runs": n_runs,
        "baseline_mean_gap": float(
            setting_df[
                "baseline_validation_test_gap"
            ].mean()
        ),
        "augmented_mean_gap": float(
            setting_df[
                "augmented_validation_test_gap"
            ].mean()
        ),
        "mean_delta_gap_augmented_minus_baseline": float(
            gap_delta.mean()
        ),
        "wilcoxon_statistic": statistic,
        "wilcoxon_p_value": p_value,
        "rank_biserial": matched_rank_biserial(
            gap_delta
        ),
    })

gap_tests_df = pd.DataFrame(
    gap_test_rows
)

gap_tests_df[
    "wilcoxon_p_holm"
] = holm_adjust(
    gap_tests_df[
        "wilcoxon_p_value"
    ].to_numpy()
)

gap_tests_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_validation_test_gap_tests.csv",
    index=False,
)


print("Training diagnostics")
display(
    round_numeric_columns(
        diagnostic_summary_df,
        decimals=5,
    )
)

print("\nValidation-test gap comparisons")
display(
    round_numeric_columns(
        gap_tests_df,
        decimals=5,
    )
)

In [ ]:
# ------------------------------------------------------------
# Exact baseline-source audit
# ------------------------------------------------------------

baseline_source_audit_df = (
    original_baseline_df
    .groupby(
        [
            "dataset_label",
            "base_seed",
        ],
        as_index=False,
    )
    .agg(
        n_subjects=("subject", "nunique"),
        mean_test_auc=("test_auc", "mean"),
        mean_validation_auc=(
            "best_val_auc",
            "mean",
        ),
        median_best_epoch=(
            "best_epoch",
            "median",
        ),
    )
)

baseline_source_audit_df.to_csv(
    FULL_TABLE_DIR
    / "original_baseline_source_audit.csv",
    index=False,
)

display(
    round_numeric_columns(
        baseline_source_audit_df,
        decimals=5,
    )
)

expected_per_dataset = {
    "ds003061": 13,
    "ds005863": 30,
    "ds006466": 30,
    "ds006480_pooled": 30,
}

for dataset_label, expected_subjects in (
        expected_per_dataset.items()
):
    dataset_rows = baseline_source_audit_df[
        baseline_source_audit_df[
            "dataset_label"
        ].eq(dataset_label)
    ]

    if len(dataset_rows) != 3:
        raise RuntimeError(
            f"{dataset_label} does not contain all three seeds."
        )

    if not dataset_rows[
        "n_subjects"
    ].eq(expected_subjects).all():
        raise RuntimeError(
            f"{dataset_label} does not contain "
            f"{expected_subjects} subjects for every seed."
        )

print(
    "The augmentation comparison uses the exact original "
    "seed-specific final EEGNet LOSO results as its baseline."
)

In [ ]:
# ------------------------------------------------------------
# Figure 1: pilot validation-only sigma selection
# ------------------------------------------------------------

if "pilot_sigma_summary_df" in globals():
    figure, axis = plt.subplots(
        figsize=(8, 5)
    )

    axis.errorbar(
        pilot_sigma_summary_df["sigma"],
        pilot_sigma_summary_df[
            "mean_val_auc_delta"
        ],
        yerr=(
                pilot_sigma_summary_df[
                    "val_delta_sd"
                ]
                / np.sqrt(
            pilot_sigma_summary_df[
                "n_paired_partitions"
            ]
        )
        ),
        marker="o",
        capsize=4,
    )

    axis.axhline(
        0,
        linestyle="--",
        linewidth=1,
    )

    axis.set_xlabel(
        "Gaussian-noise sigma after normalization"
    )

    axis.set_ylabel(
        "Validation AUC change vs baseline"
    )

    axis.set_title(
        "Validation-only augmentation-scale pilot"
    )

    figure.tight_layout()

    save_figure(
        figure,
        FIG_DIR
        / "augmentation_pilot_validation_delta.png",
        dpi=220,
    )

    plt.show()


# ------------------------------------------------------------
# Figure 2: overall subject-level AUC change
# ------------------------------------------------------------

setting_order = [
    setting
    for setting in [
        "Within-subject",
        "LOSO",
    ]
    if setting
       in subject_comparison_df[
           "evaluation_setting"
       ].unique()
]

overall_values = [
    subject_comparison_df.loc[
        subject_comparison_df[
            "evaluation_setting"
        ].eq(setting),
        "delta_auc_augmented_minus_baseline",
    ].to_numpy()
    for setting in setting_order
]

figure, axis = plt.subplots(
    figsize=(8, 5.5)
)

axis.boxplot(
    overall_values,
    tick_labels=setting_order,
    showfliers=False,
)

rng = np.random.default_rng(42)

for position, values in enumerate(
        overall_values,
        start=1,
):
    jitter = rng.normal(
        0,
        0.045,
        size=len(values),
    )

    axis.scatter(
        np.full(
            len(values),
            position,
        )
        + jitter,
        values,
        alpha=0.55,
        s=24,
    )

axis.axhline(
    0,
    linestyle="--",
    linewidth=1,
)

axis.set_ylabel(
    "Augmented minus baseline EEGNet ROC-AUC"
)

axis.set_title(
    "Subject-level effect of training-only Gaussian noise"
)

figure.tight_layout()

save_figure(
    figure,
    FIG_DIR
    / "augmentation_subject_delta_overall.png",
    dpi=220,
)

plt.show()


# ------------------------------------------------------------
# Figure 3: easy and hard subjects
# ------------------------------------------------------------

figure, axes = plt.subplots(
    1,
    len(setting_order),
    figsize=(
        6 * len(setting_order),
        5.5,
    ),
    sharey=True,
)

if len(setting_order) == 1:
    axes = [axes]

for axis, setting in zip(
        axes,
        setting_order,
):
    setting_df = quartile_df[
        quartile_df[
            "evaluation_setting"
        ].eq(setting)
    ]

    group_order = [
        "hard",
        "easy",
    ]

    values = [
        setting_df.loc[
            setting_df[
                "quality_group"
            ].eq(group),
            "delta_auc_augmented_minus_baseline",
        ].to_numpy()
        for group in group_order
    ]

    axis.boxplot(
        values,
        tick_labels=[
            "Hard",
            "Easy",
        ],
        showfliers=False,
    )

    rng = np.random.default_rng(42)

    for position, group_values in enumerate(
            values,
            start=1,
    ):
        jitter = rng.normal(
            0,
            0.045,
            size=len(group_values),
        )

        axis.scatter(
            np.full(
                len(group_values),
                position,
            )
            + jitter,
            group_values,
            alpha=0.60,
            s=28,
        )

    axis.axhline(
        0,
        linestyle="--",
        linewidth=1,
    )

    axis.set_title(setting)
    axis.set_xlabel(
        "Subject-quality group"
    )

axes[0].set_ylabel(
    "Augmented minus baseline EEGNet ROC-AUC"
)

figure.suptitle(
    "Gaussian-noise augmentation in hard and easy subjects"
)

figure.tight_layout()

save_figure(
    figure,
    FIG_DIR
    / "augmentation_subject_delta_easy_hard.png",
    dpi=220,
)

plt.show()


# ------------------------------------------------------------
# Figure 4: validation-test generalization gap
# ------------------------------------------------------------

gap_plot_df = diagnostic_summary_df.copy()

figure, axis = plt.subplots(
    figsize=(9, 5.5)
)

x_labels = []

x_positions = []

gap_values = []

for setting_index, setting in enumerate(
        setting_order
):
    for condition_index, condition in enumerate(
            [
                "baseline",
                "gaussian_noise",
            ]
    ):
        row = gap_plot_df[
            gap_plot_df[
                "evaluation_setting"
            ].eq(setting)
            & gap_plot_df[
                "condition"
            ].eq(condition)
            ]

        if row.empty:
            continue

        position = (
                setting_index * 3
                + condition_index
        )

        x_positions.append(
            position
        )

        x_labels.append(
            f"{setting}\n{condition}"
        )

        gap_values.append(
            float(
                row.iloc[0][
                    "mean_validation_test_gap"
                ]
            )
        )

axis.bar(
    x_positions,
    gap_values,
)

axis.axhline(
    0,
    linewidth=1,
)

axis.set_xticks(
    x_positions,
)

axis.set_xticklabels(
    x_labels,
)

axis.set_ylabel(
    "Mean validation AUC minus test AUC"
)

axis.set_title(
    "EEGNet validation-test gap by condition"
)

figure.tight_layout()

save_figure(
    figure,
    FIG_DIR
    / "augmentation_validation_test_gap.png",
    dpi=220,
)

plt.show()

In [ ]:
# ------------------------------------------------------------
# Generate conservative interpretation guidance
# ------------------------------------------------------------

interpretation_lines = []

interpretation_lines.append(
    "EEGNet Gaussian-noise augmentation ablation"
)

interpretation_lines.append(
    "=" * 70
)

if FROZEN_CHOICE_PATH.exists():
    with FROZEN_CHOICE_PATH.open(
            "r",
            encoding="utf-8",
    ) as file:
        frozen_choice = json.load(
            file
        )

    interpretation_lines.append(
        "Frozen sigma: "
        f"{frozen_choice['frozen_sigma']}"
    )

interpretation_lines.append("")

for _, row in overall_statistics_df.iterrows():
    setting = row[
        "evaluation_setting"
    ]

    mean_delta = row[
        "mean_delta"
    ]

    p_adjusted = row[
        "wilcoxon_p_holm"
    ]

    ci_low = row[
        "bootstrap_mean_delta_95ci_low"
    ]

    ci_high = row[
        "bootstrap_mean_delta_95ci_high"
    ]

    interpretation_lines.append(
        f"{setting}:"
    )

    interpretation_lines.append(
        f"  Baseline mean AUC: "
        f"{row['baseline_mean_auc']:.4f}"
    )

    interpretation_lines.append(
        f"  Augmented mean AUC: "
        f"{row['augmented_mean_auc']:.4f}"
    )

    interpretation_lines.append(
        f"  Mean AUC change: "
        f"{mean_delta:+.4f}"
    )

    interpretation_lines.append(
        f"  Bootstrap 95% CI: "
        f"[{ci_low:+.4f}, {ci_high:+.4f}]"
    )

    interpretation_lines.append(
        f"  Holm-adjusted paired p: "
        f"{p_adjusted:.6g}"
    )

    interpretation_lines.append(
        f"  Proportion improved: "
        f"{row['proportion_improved']:.3f}"
    )

    if (
            mean_delta > 0
            and p_adjusted < 0.05
            and ci_low > 0
    ):
        interpretation_lines.append(
            "  Interpretation: augmentation produced a "
            "statistically supported positive mean effect."
        )

    elif (
            mean_delta < 0
            and p_adjusted < 0.05
            and ci_high < 0
    ):
        interpretation_lines.append(
            "  Interpretation: augmentation produced a "
            "statistically supported negative mean effect."
        )

    else:
        interpretation_lines.append(
            "  Interpretation: no statistically supported "
            "overall improvement was demonstrated."
        )

    interpretation_lines.append("")


for _, row in quality_group_tests_df.iterrows():
    interpretation_lines.append(
        f"{row['evaluation_setting']} — "
        f"{row['quality_group']} subjects:"
    )

    interpretation_lines.append(
        f"  Mean change: "
        f"{row['mean_delta']:+.4f}"
    )

    interpretation_lines.append(
        f"  Holm-adjusted paired p: "
        f"{row['wilcoxon_p_holm']:.6g}"
    )

    interpretation_lines.append(
        f"  Proportion improved: "
        f"{row['proportion_improved']:.3f}"
    )

    interpretation_lines.append("")


interpretation_text = "\n".join(
    interpretation_lines
)

print(interpretation_text)

interpretation_path = (
        FULL_TABLE_DIR
        / "augmentation_interpretation_helper.txt"
)

interpretation_path.write_text(
    interpretation_text,
    encoding="utf-8",
)

print(
    "\nSaved interpretation helper:",
    interpretation_path,
)

In [ ]:
# ------------------------------------------------------------
# Subject-level validation-test gap analysis
# Average seeds before statistical inference
# ------------------------------------------------------------

gap_subject_df = (
    outer_comparison_df
    .groupby(
        [
            "evaluation_setting",
            "dataset_label",
            "subject",
        ],
        as_index=False,
    )
    .agg(
        baseline_validation_test_gap=(
            "baseline_validation_test_gap",
            "mean",
        ),
        augmented_validation_test_gap=(
            "augmented_validation_test_gap",
            "mean",
        ),
        delta_gap_augmented_minus_baseline=(
            "delta_gap_augmented_minus_baseline",
            "mean",
        ),
    )
)

if len(gap_subject_df) != 103:
    raise RuntimeError(
        "Expected 103 subject-level gap comparisons, "
        f"but found {len(gap_subject_df)}."
    )


gap_test_rows = []

for evaluation_setting, setting_df in (
        gap_subject_df.groupby("evaluation_setting")
):
    delta = setting_df[
        "delta_gap_augmented_minus_baseline"
    ].to_numpy(dtype=float)

    statistic, p_value, n_subjects = safe_wilcoxon(
        delta
    )

    ci_low, ci_high = bootstrap_mean_ci(
        delta,
        seed=42,
    )

    gap_test_rows.append({
        "evaluation_setting": evaluation_setting,
        "n_subjects": int(n_subjects),
        "baseline_mean_gap": float(
            setting_df[
                "baseline_validation_test_gap"
            ].mean()
        ),
        "augmented_mean_gap": float(
            setting_df[
                "augmented_validation_test_gap"
            ].mean()
        ),
        "mean_delta_gap_augmented_minus_baseline": float(
            delta.mean()
        ),
        "bootstrap_mean_delta_95ci_low": ci_low,
        "bootstrap_mean_delta_95ci_high": ci_high,
        "wilcoxon_statistic": statistic,
        "wilcoxon_p_value": p_value,
        "rank_biserial": matched_rank_biserial(
            delta
        ),
    })


gap_tests_df = pd.DataFrame(
    gap_test_rows
)

gap_tests_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_validation_test_gap_tests.csv",
    index=False,
)

gap_subject_df.to_csv(
    FULL_TABLE_DIR
    / "augmentation_validation_test_gap_subjects.csv",
    index=False,
)

display(
    round_numeric_columns(
        gap_tests_df,
        decimals=6,
    )
)

In [ ]:
# ------------------------------------------------------------
# Save final artifact manifest
# ------------------------------------------------------------

manifest_rows = []

for category, directory, pattern in [
    (
            "pilot_table",
            PILOT_TABLE_DIR,
            "*.csv",
    ),
    (
            "pilot_history",
            PILOT_HISTORY_DIR,
            "*.csv",
    ),
    (
            "full_table_csv",
            FULL_TABLE_DIR,
            "*.csv",
    ),
    (
            "full_table_text",
            FULL_TABLE_DIR,
            "*.txt",
    ),
    (
            "full_history",
            FULL_HISTORY_DIR,
            "*.csv",
    ),
    (
            "figure",
            FIG_DIR,
            "*.png",
    ),
    (
            "error",
            ERROR_DIR,
            "*.json",
    ),
]:
    for path in sorted(
            directory.glob(pattern)
    ):
        manifest_rows.append({
            "category": category,
            "name": path.name,
            "path": str(path),
            "size_bytes": int(
                path.stat().st_size
            ),
        })

if FROZEN_CHOICE_PATH.exists():
    manifest_rows.append({
        "category": "frozen_choice",
        "name": FROZEN_CHOICE_PATH.name,
        "path": str(FROZEN_CHOICE_PATH),
        "size_bytes": int(
            FROZEN_CHOICE_PATH.stat().st_size
        ),
    })

manifest_df = pd.DataFrame(
    manifest_rows
)

manifest_df.to_csv(
    OUT_ROOT
    / "augmentation_artifact_manifest.csv",
    index=False,
)

print("=" * 90)
print("Augmentation notebook outputs")
print("=" * 90)

print("\nPilot tables:")
for path in sorted(
        PILOT_TABLE_DIR.glob("*.csv")
):
    print(" -", path)

print("\nFull tables:")
for path in sorted(
        FULL_TABLE_DIR.glob("*")
):
    if path.is_file():
        print(" -", path)

print("\nFigures:")
for path in sorted(
        FIG_DIR.glob("*.png")
):
    print(" -", path)

print("\nHistories:")
print(
    "Pilot:",
    len(
        list(
            PILOT_HISTORY_DIR.glob(
                "*.csv"
            )
        )
    ),
)

print(
    "Full:",
    len(
        list(
            FULL_HISTORY_DIR.glob(
                "*.csv"
            )
        )
    ),
)

print("\nErrors:")
print(
    len(
        list(
            ERROR_DIR.glob(
                "*.json"
            )
        )
    )
)

print("\nManifest saved:")
print(
    OUT_ROOT
    / "augmentation_artifact_manifest.csv"
)

## Final completion checklist

Before using the augmentation result in the dissertation or response letter, verify
all of the following.

### Experimental completeness

- `frozen_augmentation_choice.json` was created before the full run.
- The frozen sigma was not changed after viewing outer-test results.
- Every full outer partition contains one baseline and one augmented condition.
- The complete run contains 1,839 paired partition-seed observations:
  - 1,530 grouped within-subject;
  - 309 LOSO.
- The condition-level table contains 3,678 rows.
- No validation or test batch was augmented.
- No failed run remains unresolved.

### Baseline agreement

- Compare the rerun baseline against the original final EEGNet subject-level AUC.
- Investigate any material discrepancy before interpreting augmentation.
- Confirm that the same subject lists, folds, purge settings, crop, seeds,
  normalization, architecture, and early-stopping rules were used.

### Statistical interpretation

- Use subject-level AUC averages, not fold rows, for the primary paired tests.
- Report mean and median AUC change.
- Report the 95% bootstrap confidence interval.
- Report the paired Wilcoxon test and matched rank-biserial effect size.
- Report the proportion of subjects whose AUC improved.
- Report hard and easy subjects separately.
- Report the hard-versus-easy comparison.
- Report the continuous Subject Quality Index versus augmentation-gain correlation.
- Treat quartile analyses as retrospective and exploratory.

### Dissertation framing

The original non-augmented benchmark remains the principal benchmark.

The augmentation experiment must be described as:

> a supplementary post-defense training-only robustness ablation conducted in
> response to the examiner's concern about neural-model overfitting.

Possible conclusions must follow the result:

- **Positive result:** Gaussian-noise augmentation modestly improved EEGNet
  generalization, with the exact magnitude and affected evaluation setting stated.
- **Null result:** small unstructured perturbations did not materially improve
  generalization or solve the hard-subject problem.
- **Negative result:** added unstructured noise weakened already low-SNR ERP
  information.